In [2]:
import pandas as pd

In [3]:
file_path = "yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(
    file_path,
    nrows=100
)

print(df.head())

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1         1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2         1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3         1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4         2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           2.10           1                  N           142            43   
1           0.20           1                  N           238           151   
2          14.70           1                  N           132           165   
3          10.60           1                  N           138           132   
4           4.94           1                  N            68            33   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


In [4]:
print(df.dtypes)

VendorID                   int64
tpep_pickup_datetime         str
tpep_dropoff_datetime        str
passenger_count            int64
trip_distance            float64
RatecodeID                 int64
store_and_fwd_flag           str
PULocationID               int64
DOLocationID               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount             float64
congestion_surcharge     float64
dtype: object


In [5]:
print(df.isnull().sum())

VendorID                 0
tpep_pickup_datetime     0
tpep_dropoff_datetime    0
passenger_count          0
trip_distance            0
RatecodeID               0
store_and_fwd_flag       0
PULocationID             0
DOLocationID             0
payment_type             0
fare_amount              0
extra                    0
mta_tax                  0
tip_amount               0
tolls_amount             0
improvement_surcharge    0
total_amount             0
congestion_surcharge     0
dtype: int64


In [6]:
import pandas as pd

file_path = "yellow_tripdata_2021-01.csv.gz"

chunk_size = 100000

total_rows = 0

null_counts = None

min_values = {}

max_values = {}

dtypes = {}

for chunk in pd.read_csv(
    file_path,
    chunksize=chunk_size,
    low_memory=False
):

    total_rows += len(chunk)

    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()

    for col in chunk.columns:

        dtypes[col] = str(chunk[col].dtype)

        col_min = chunk[col].min()
        col_max = chunk[col].max()

        if col not in min_values:
            min_values[col] = col_min
            max_values[col] = col_max

        else:
            if pd.notna(col_min):
                if pd.isna(min_values[col]) or col_min < min_values[col]:
                    min_values[col] = col_min

            if pd.notna(col_max):
                if pd.isna(max_values[col]) or col_max > max_values[col]:
                    max_values[col] = col_max

print("Total rows:", total_rows)

Total rows: 1369765


In [7]:
profile = pd.DataFrame({
    "dtype": pd.Series(dtypes),
    "null_count": null_counts,
    "null_percent": (null_counts / total_rows) * 100,
    "min": pd.Series(min_values),
    "max": pd.Series(max_values)
})

print(profile.to_string())

                         dtype  null_count  null_percent                  min                  max
VendorID               float64       98352       7.18021                    1                    2
tpep_pickup_datetime       str           0       0.00000  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime      str           0       0.00000  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count        float64       98352       7.18021                    0                    8
trip_distance          float64           0       0.00000                  0.0            263163.28
RatecodeID             float64       98352       7.18021                    1                   99
store_and_fwd_flag     float64       98352       7.18021                    N                    Y
PULocationID             int64           0       0.00000                    1                  265
DOLocationID             int64           0       0.00000                    1                  265
payment_ty

In [8]:
bad_dates = 0
bad_fares = 0
bad_distances = 0
zero_passengers = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=100000,
    low_memory=False
):

    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"],
        errors="coerce"
    )

    bad_dates += (
        (pickup < "2021-01-01") |
        (pickup >= "2021-02-01") |
        pickup.isna()
    ).sum()

    bad_fares += (
        chunk["fare_amount"] <= 0
    ).sum()

    bad_distances += (
        chunk["trip_distance"] <= 0
    ).sum()

    zero_passengers += (
        chunk["passenger_count"] == 0
    ).sum()

In [9]:
problems = pd.DataFrame({
    "Problem": [
        "Pickup outside January 2021",
        "Fare <= 0",
        "Trip distance <= 0",
        "Passenger count = 0"
    ],

    "Count": [
        bad_dates,
        bad_fares,
        bad_distances,
        zero_passengers
    ]
})

print(problems.to_string(index=False))

                    Problem  Count
Pickup outside January 2021     24
                  Fare <= 0   7411
         Trip distance <= 0  19952
        Passenger count = 0  26726


In [10]:
import hashlib

seen = set()
duplicate_count = 0

for chunk in pd.read_csv(
    file_path,
    chunksize=100000,
    low_memory=False
):

    for row in chunk.itertuples(index=False, name=None):

        row_bytes = repr(row).encode("utf-8")

        row_hash = hashlib.sha256(row_bytes).digest()

        if row_hash in seen:
            duplicate_count += 1
        else:
            seen.add(row_hash)

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


In [12]:
def clean(df):

    cleaned = df.copy()

    rows_removed = {}

    # Rule 1: Remove invalid pickup dates

    pickup = pd.to_datetime(
        cleaned["tpep_pickup_datetime"],
        errors="coerce"
    )

    valid_dates = (
        (pickup >= "2021-01-01") &
        (pickup < "2021-02-01")
    )

    rows_removed["invalid_pickup_date"] = int(
        (~valid_dates).sum()
    )

    cleaned = cleaned.loc[valid_dates].copy()

    # Rule 2: Remove exact duplicate rows

    duplicates = cleaned.duplicated()

    rows_removed["duplicate_rows"] = int(
        duplicates.sum()
    )

    cleaned = cleaned.loc[~duplicates].copy()

    # Rule 3: Flag unusual fares

    cleaned["flag_bad_fare"] = (
        cleaned["fare_amount"] <= 0
    )

    rows_removed["bad_fare"] = 0

    # Rule 4: Flag unusual distances

    cleaned["flag_bad_distance"] = (
        cleaned["trip_distance"] <= 0
    )

    rows_removed["bad_distance"] = 0

    # Rule 5: Flag zero passengers

    cleaned["flag_zero_passengers"] = (
        cleaned["passenger_count"] == 0
    )

    rows_removed["zero_passengers"] = 0

    return cleaned, rows_removed

In [13]:
cleaned_df, removed = clean(df)

print("Rows before:", len(df))

print("Rows after:", len(cleaned_df))

print("Rows removed per rule:")

print(pd.DataFrame(
    list(removed.items()),
    columns=["Rule", "Rows Removed"]
).to_string(index=False))

Rows before: 100
Rows after: 100
Rows removed per rule:
               Rule  Rows Removed
invalid_pickup_date             0
     duplicate_rows             0
           bad_fare             0
       bad_distance             0
    zero_passengers             0


In [20]:
print("Original rows:", len(df))

print("Cleaned rows:", len(cleaned_df))

print("Total removed:", sum(removed.values()))

print(
    "Original unchanged:",
    df.equals(pd.read_csv(file_path, nrows=100))
)

Original rows: 100
Cleaned rows: 100
Total removed: 0
Original unchanged: True


In [21]:
pickup = pd.to_datetime(
    cleaned_df["tpep_pickup_datetime"],
    errors="coerce"
)

print(
    "Invalid dates remaining:",
    (
        (pickup < "2021-01-01") |
        (pickup >= "2021-02-01") |
        pickup.isna()
    ).sum()
)

print(
    "Duplicates remaining:",
    cleaned_df.duplicated(
        subset=df.columns.tolist()
    ).sum()
)

Invalid dates remaining: 0
Duplicates remaining: 0
